# Fase 3: Agente con RAG

## 1. Código heredado de la Fase 2

In [1]:
import os
import numpy as np
from dotenv import load_dotenv
from qdrant_client import models, QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, SparseVector
from sentence_transformers import CrossEncoder, SentenceTransformer
from fastembed import SparseTextEmbedding

# Conectar con Qdrant
cliente_qdrant = QdrantClient(host="localhost", port=6333)

# Inicializar los modelos de embedding denso y sparse para retrieval
modelo_emb = SentenceTransformer("intfloat/multilingual-e5-base")
sparse_model = SparseTextEmbedding("Qdrant/bm25")

# Inicializar el cross-encoder para reranking
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3")

c:\Users\andre\Documents\GitHub\PECL_LLMs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3675.73it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Users\andre\Documents\GitHub\PECL_LLMs\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 4626.68it/s]


In [2]:
def buscar_qdrant(query, top_k_inicial=10, top_k_final=3, filtro_categoria=None, filtro_empresa=None, mostrar_chunks=False):
    """
    Realiza una búsqueda en Qdrant utilizando un pipeline completo de RAG:

    1. Búsqueda híbrida (dense + sparse/BM25):
       - Combina embeddings densos (E5) con representaciones sparse (BM25)
       - Los resultados de ambas búsquedas se fusionan mediante Reciprocal Rank Fusion (RRF)
       - Mejora la recuperación al capturar similitud semántica y coincidencias léxicas

    2. Filtrado por metadatos (opcional):
       - Permite restringir resultados por categoría o empresa

    3. Reranking con cross-encoder:
       - Reordena los resultados iniciales evaluando directamente la relación (query, chunk)
       - Mejora la precisión final del contexto

    Parameters:
        query (str): consulta del usuario
        top_k_inicial (int): número de candidatos recuperados inicialmente
        top_k_final (int): número de resultados tras reranking
        filtro_categoria (str, opcional): filtrar por categoría
        filtro_empresa (str, opcional): filtrar por empresa

    Returns:
        list[dict]: lista de chunks relevantes con metadata y scores
    """
    # --- 1. RETRIEVAL HÍBRIDO (dense + sparse)
    # Embedding denso (semántico)
    query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]
    
    # Embedding disperso (coincidencia léxica)
    query_sparse_raw = list(sparse_model.embed([query]))[0]
    query_sparse = SparseVector(
        indices=query_sparse_raw.indices.tolist(),
        values=query_sparse_raw.values.tolist(),
    )

    # FILTRADO POR METADATA (opcional)
    filtro = None

    if filtro_categoria or filtro_empresa:
        must_conditions = []
        
        # Categorías (OR entre varias categorías)
        if filtro_categoria:
            if isinstance(filtro_categoria, list):
                # Si es una lista de categorías, se crea un bloque OR (should) para ellas
                categorias_cond = Filter(
                    should=[
                        FieldCondition(
                            key="categoria",
                            match=MatchValue(value=cat)
                        )
                        for cat in filtro_categoria
                    ]
                )
                must_conditions.append(categorias_cond)
            else:
                # Si es una única categoría, se añade directamente como condición
                must_conditions.append(
                    FieldCondition(
                        key="categoria",
                        match=MatchValue(value=filtro_categoria)
                    )
                )

        # Empresas (OR entre varias empresas)
        # Solo se aplica este filtro si la categoría es "terms" o "privacy_policies"
        if filtro_empresa:
            aplicar_filtro_empresa = False

            if isinstance(filtro_categoria, list):
                if "guides" in filtro_categoria:
                    aplicar_filtro_empresa = False
                else:
                    aplicar_filtro_empresa = any(
                        cat in ["terms", "privacy_policies"] for cat in filtro_categoria
                    )
            else:
                aplicar_filtro_empresa = filtro_categoria in ["terms", "privacy_policies"]

            if aplicar_filtro_empresa:
                if isinstance(filtro_empresa, list):
                    empresas_cond = Filter(
                        should=[
                            FieldCondition(
                                key="empresa",
                                match=MatchValue(value=emp)
                            )
                            for emp in filtro_empresa
                        ]
                    )
                    must_conditions.append(empresas_cond)
                else:
                    must_conditions.append(
                        FieldCondition(
                            key="empresa",
                            match=MatchValue(value=filtro_empresa)
                        )
                    )

        # Construcción final (AND entre bloques)
        filtro = Filter(must=must_conditions)

    # --- BÚSQUEDA EN QDRANT (HÍBRIDA)
    # Qdrant combina automáticamente dense + sparse en un único score híbrido
    resultados = cliente_qdrant.query_points(
        collection_name="rag_privacy_corpus",
        prefetch=[
            # Recupera candidatos de forma independiente para cada tipo de representación
            models.Prefetch(
                query=query_emb.tolist(),
                using="dense",
                limit=top_k_inicial * 2, # Doble de candidatos para mejorar recall antes del reranking
            ),
            models.Prefetch(
                query=query_sparse,
                using="sparse",
                limit=top_k_inicial * 2,
            ),
        ],
        # Combina los rankings de dense y sparse usando RRF
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        # Se reduce el número de candidatos antes del reranking
        limit=top_k_inicial, 
        query_filter=filtro,
        with_payload=True,
    )

    # --- 2. RERANKING: el cross-encoder evalúa cada par (query, chunk) conjuntamente
    # Crear pares (query, chunk)
    pairs = [
        (query, punto.payload["chunk"])
        for punto in resultados.points
    ]

    # Obtener scores del reranker
    scores = reranker.predict(pairs)

    # Combinar resultados
    reranked = []
    for i, punto in enumerate(resultados.points):
        reranked.append({
            "chunk": punto.payload["chunk"],
            "categoria": punto.payload["categoria"],
            "archivo": punto.payload["archivo"],
            "empresa": punto.payload.get("empresa", None),
            # Score híbrido (dense + sparse combinado por Qdrant)
            "score_hybrid": punto.score,
            # Score del reranker (decisión final)
            "score_rerank": float(scores[i])
        })

    # Ordenar por score del reranker
    reranked = sorted(reranked, key=lambda x: x["score_rerank"], reverse=True)

    # Seleccionar top final
    reranked_final = reranked[:top_k_final]

    # --- DEBUG / PRINT
    if mostrar_chunks:
        print(f"Query: '{query}'")
        if filtro_categoria:
            print(f"Filtro: categoria = '{filtro_categoria}'")
        if filtro_empresa:
            print(f"Filtro: empresa = '{filtro_empresa}'")
        print()

        for i, r in enumerate(reranked_final):
            print(f"{i + 1}. (rerank={r['score_rerank']:.3f} | hybrid={r['score_hybrid']:.3f}) [{r['categoria']} - {r['archivo']}]")
            print(f"   {r['chunk']}\n")

    return reranked_final

In [3]:
# --- FUNCIONES AUXILIARES PARA DETECCIÓN DE FILTROS
def detectar_empresas(query):
    empresas = []
    
    query_lower = query.lower()
    if "google" in query_lower:
        empresas.append("Google")
    if "spotify" in query_lower:
        empresas.append("Spotify")

    return empresas if empresas else None
    
# Descripciones para cada categoría
categoria_descripciones = {
    "guides": "explanations of privacy concepts like GDPR and cookies",
    "privacy_policies": "how companies collect and use personal data",
    "terms": "terms of service and usage conditions"
}

# Generar embeddings para cada descripción
categoria_embeddings = {}
for cat, desc in categoria_descripciones.items():
    emb = modelo_emb.encode(["passage: " + desc], normalize_embeddings=True)[0]
    categoria_embeddings[cat] = emb
    
def detectar_categorias(query, margin=0.02, min_score=0.20):
    query_emb = modelo_emb.encode(["query: " + query], normalize_embeddings=True)[0]

    # Calcular similitud coseno entre la query y cada descripción de categoría
    scores = {}
    for cat, desc_emb in categoria_embeddings.items():
        scores[cat] = np.dot(query_emb, desc_emb)

    # Umbral mínimo real: si no hay suficiente similitud, no aplicar filtro de categoría
    max_score = max(scores.values())
    if max_score < min_score:
        return None

    # Identificar categorías con score cercano al máximo (dentro de un margen)
    categorias = [
        cat for cat, score in scores.items()
        if (max_score - score) <= margin
    ]

    return categorias if categorias else None

## 2. Creación del Agente con Tools

In [4]:
# Jupyter ya tiene un event loop corriendo. nest_asyncio permite que pydantic-ai lance su propio loop dentro del de Jupyter.
import nest_asyncio

nest_asyncio.apply()

### 2.1. Agente con la Tool de búsqueda en Qdrant

In [6]:
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.groq import GroqModel
from pydantic_ai.providers.groq import GroqProvider
from pydantic import BaseModel, Field

# Cargar la clave de API desde el archivo .env
load_dotenv()
API_KEY = os.getenv("GROQ_KEY")

# Inicializar el modelo de Groq con la clave de API
model = GroqModel(
    "openai/gpt-oss-120b", 
    provider=GroqProvider(api_key=API_KEY),
    settings={
        "temperature": 0.2
    }
)

# Prompt de sistema similar al de la fase 2 pero adaptado para el Agente de RAG con tools
SYSTEM_PROMPT_AGENT = (
    "Eres un asistente especializado en privacidad, protección de datos y términos de servicio.\n\n"

    "Dispones de una herramienta llamada 'search_documents' que permite buscar información en documentos oficiales.\n"
    "Dispones de una herramienta llamada 'web_search' que permite buscar información en internet.\n"
    "Tambien, tienes acceso a otras herramientas auxiliares.\n\n"

    "INSTRUCCIONES DE USO DE HERRAMIENTAS:\n"
    "- Si la pregunta requiere información específica, normativa o detallada del dominio, usa la herramienta 'search_documents'.\n"
    "- Si no estas seguro de tener información oficial y verificada, usa la herramienta 'search_documents'.\n"
    "- Si puedes responder con conocimiento general fiable, responde directamente sin usar la herramienta.\n"
    "- Usa la herramienta 'web_search' SOLO cuando la herramienta 'search_documents' no devuelva resultados suficientes para responder a la pregunta del usuario.\n\n"

    "USO DE LA INFORMACIÓN:\n"
    "- Cuando uses información de la herramienta 'search_documents', intégrala de forma natural en la respuesta.\n"
    "- No menciones herramientas, contexto, fragmentos ni estructuras internas del corpus de documentos.\n"
    "- Puedes referirte a 'documentos oficiales' de forma general si es necesario cuando uses 'search_documents'.\n\n"

    "RESTRICCIONES:\n"
    "- No respondas preguntas fuera del dominio de privacidad y protección de datos.\n"
    "- No inventes información bajo ninguna circunstancia.\n\n"

    "FORMATO DE RESPUESTA:\n"
    "- Usa un tono formal.\n"
    "- Sé claro, breve y estructurado.\n"
    "- Responde en el mismo idioma que la pregunta del usuario."
)

agent = Agent(model, system_prompt=SYSTEM_PROMPT_AGENT)

# Input de la tool
class SearchDocumentsInput(BaseModel):
    query: str = Field(
        description=(
            "Consulta breve optimizada para búsqueda semántica. "
            "Usa palabras clave (no preguntas completas) e incluye conceptos relevantes "
            "y, si aplica, la empresa (Google, Spotify). "
            "Ejemplo: 'Spotify privacy policy personal data collection'."
        )
    )

    categoria: list[str] | None = Field(
        default=None,
        description="Filtros de categoria de documento: ['guides', 'privacy_policies', 'terms']"
    )

    empresa: list[str] | None = Field(
        default=None,
        description="Filtros de empresas relevantes (ej: ['Google', 'Spotify'])"
    )
    

# Tool para buscar contexto en Qdrant, a diferencia de la fase 2, ahora permitimos que 
# el agente puede decidir qué filtros de categoría y empresa usar.
@agent.tool
def search_documents(ctx: RunContext, input: SearchDocumentsInput) -> str:
    """
    Busca información en un corpus de documentos oficiales y verificados sobre privacidad y protección de datos.

    El corpus incluye tres tipos de contenido:
    (1) guías explicativas sobre conceptos como GDPR, cookies y derechos de los usuarios.
    (2) políticas de privacidad que describen cómo empresas como Google y Spotify recopilan, usan y comparten datos personales.
    (3) términos de servicio con condiciones de uso de plataformas como Google y Spotify.

    Usa esta herramienta cuando la pregunta requiera información específica, detallada o normativa de estos documentos.
    Formula la query como una búsqueda semántica breve incluyendo conceptos clave.
    Puedes especificar filtros opcionales para mejorar la precisión de la búsqueda según la pregunta del usuario:
        - categoria: filtros disponibles (guides, privacy_policies, terms), puedes seleccionar varias categorías si es necesario.
        - empresa: filtros disponibles (Google, Spotify), solo aplicable para categorías "terms" y "privacy_policies"
    """
    try:
        resultados = buscar_qdrant(
            input.query,
            # Si el agente no especifica filtros, se intenta detectar automáticamente 
            # a partir de la query para mejorar la búsqueda
            filtro_empresa = input.empresa or detectar_empresas(input.query),
            filtro_categoria = input.categoria or detectar_categorias(input.query),
        )

        if not resultados:
            return "NO_RESULTS"

        # Añadir nombre de documento al principio de cada chunk para mayor trazabilidad
        return "\n\n".join(
            f"[{r['archivo']}] {r['chunk']}"
            for r in resultados
        )
    
    except Exception:
        return "ERROR"

### 2.2. Tools adicionales

#### 2.2.1. Búsqueda Web mediante Tavily

In [ ]:
from tavily import TavilyClient

load_dotenv()
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Cliente para búsqueda en la web con Tavily
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

class WebSearchInput(BaseModel):
    query: str = Field(
        description="Consulta breve para buscar información en la web"
    )

@agent.tool
def web_search(ctx: RunContext, input: WebSearchInput) -> str:
    """
    Busca información en internet.

    Usa esta herramienta SOLO cuando la herramienta search_documents no devuelva resultados suficientes.
    """
    try:
        response = tavily_client.search(
            query=input.query,
            max_results=5
        )

        results = response.get("results", [])

        if not results:
            return "NO_RESULTS"

        # Construir contexto limpio
        return "\n\n".join(
            f"Fuente: {r['url']}\n{r['title']}\n{r['content']}"
            for r in results
        )

    except Exception:
        return "ERROR"

#### 2.2.2. Obtener Fecha Actual

In [10]:
from datetime import datetime

class GetCurrentDateOutput(BaseModel):
    current_date: str = Field(
        description="Fecha actual en formato YYYY-MM-DD"
    )

@agent.tool
def get_current_date(ctx: RunContext) -> GetCurrentDateOutput:
    """
    Devuelve la fecha actual.

    Usa esta herramienta cuando la pregunta requiera información temporal
    o dependa de la fecha actual.
    """
    try:
        return GetCurrentDateOutput(
            current_date=datetime.now().strftime("%Y-%m-%d")
        )
    except Exception:
        return GetCurrentDateOutput(current_date="ERROR")

#### 2.2.3. Calculadora

In [11]:
class CalculatorInput(BaseModel):
    expression: str = Field(
        description="Expresión matemática simple (ej: '1000 * 0.04')"
    )

@agent.tool
def calculator(ctx: RunContext, input: CalculatorInput) -> str:
    """
    Realiza cálculos matemáticos.

    Usa esta herramienta para operaciones numéricas precisas,
    como porcentajes o cálculos relacionados con multas o límites.
    """
    try:
        # Evaluación controlada (simple)
        expression = input.expression.replace(" ", "")
        
        # Solo permitir caracteres válidos
        allowed_chars = "0123456789+-*/().%"
        if any(c not in allowed_chars for c in expression):
            return "ERROR"

        result = eval(expression)
        return str(result)

    except Exception:
        return "ERROR"

## 3. Probar el funcionamiento del Agente RAG

In [12]:
from pydantic_ai.messages import ToolCallPart, ToolReturnPart, TextPart, ThinkingPart

def responder_con_trazas(pregunta):
    result = agent.run_sync(pregunta)
    print("Respuesta:", result.output)
    print("=" * 70)

    for msg in result.all_messages():
        for part in msg.parts:
            if isinstance(part, ToolCallPart):
                print(f"> Action:      {part.tool_name}({part.args})")
            elif isinstance(part, ToolReturnPart):
                print(f"> Observation: {str(part.content)}")
            elif isinstance(part, ThinkingPart):
                print(f"> Thinking:    {part.content}")
            elif isinstance(part, TextPart):
                print(f"> Respuesta:   {part.content}")

### 3.1. Preguntas generales dentro del dominio

Las siguientes pruebas permiten analizar cómo el agente decide autónomamente cuándo utilizar herramientas y cómo reformula las búsquedas respecto a la pregunta original del usuario.

In [15]:
responder_con_trazas("¿Que son las cookies y para qué se utilizan?")
# Si se ejecuta varias veces, la mayoria de ellas el agente usa la tool, 
# pero otras veces, responde directamente porque tiene suficiente conocimiento general para responder.

Respuesta: Las cookies son archivos de texto que almacenan pequeñas cantidades de datos en el dispositivo del usuario para identificarlo durante la navegación en internet. Su principal función es mejorar la experiencia del usuario al recordar preferencias, mantener sesiones activas (como credenciales de inicio de sesión) y personalizar el contenido. 

### Tipos y usos comunes:
1. **Cookies esenciales**: Necesarias para el funcionamiento básico de un sitio web (ej.: mantener el usuario autenticado).
2. **Cookies de terceros**: Utilizadas por servicios externos (como redes sociales o analíticas) para rastrear comportamientos y publicidad.
3. **Cookies "zombie"**: Persistentes y difíciles de eliminar, pueden reemergir tras ser borradas, usándose para seguimiento o, en casos extremos, para malware.

### Regulación:
Leyes como el **GDPR** (UE) y el **CCPA** (California) exigen que los sitios soliciten permiso explícito antes de usar cookies no esenciales, informando sobre su finalidad. Los 

In [57]:
responder_con_trazas("¿Qué derechos tienen los usuarios sobre sus datos personales?")
# En el razonamiento, el agente indica que tiene conocimiento general, pero prefiere usar la herramienta 
# para asegurar que la información es oficial y verificada, lo cual es correcto dada la naturaleza de la pregunta.
#
# Además, se observa que los filtros que aplica en la busqueda son más precisos que los que se detectaban 
# automáticamente en la fase 2: antes se filtraba tambien por la categoria 'privacy_policies' lo que hacia que 
# no se obtuvieran chunks de 'gdpr_explained.md', que es el documento realmente relevante.

Respuesta: Los usuarios tienen derechos amplios sobre sus datos personales según normativas como el GDPR (Reglamento General de Protección de Datos). Estos derechos incluyen:

1. **Derecho a ser informados**  
   A recibir transparencia clara sobre qué datos se recopilan, cómo se usan y a quiénes se comparten.

2. **Derecho de acceso**  
   A solicitar copias de sus datos personales y verificar su tratamiento por parte de organizaciones.

3. **Derecho de rectificación**  
   A corregir errores o inexactitudes en sus datos almacenados.

4. **Derecho de supresión**  
   A borrar sus datos cuando no sean necesarios, ilegales, innecesarios o ya no sean solicitados.

5. **Derecho a limitar el tratamiento**  
   A restringir el uso de sus datos en circunstancias como disputas sobre exactitud o uso no consentido.

6. **Derecho a la portabilidad de datos**  
   A obtener sus datos y transferirlos a otro proveedor si se cumple con estándares estructurados y legibles.

7. **Derecho de oposición*

In [60]:
responder_con_trazas("¿Qué datos recopilan las empresas cuando usamos sus servicios?")
# El modelo decide usar la tool `search_documents` y selecciona filtros correctos.
# En la fase 2, al hacer esta pregunta, solo se recuperaban chunks de Spotify por que la pregunta estaba en español,
# mientras que ahora paso lo contrario: solo se recuperan chunks de Google. 
# Esto se debe a que la query de búsqueda que ha usado el modelo esta en inglés, por lo que como ya vimos en la fase 2, 
# esto favorece que obtengan mayor score los chunks de Google porque estan en inglés.

Respuesta: Las empresas recopilan diversos tipos de datos cuando utilizamos sus servicios, según su finalidad y normas de privacidad. Basado en documentos oficiales:

**Tipos de datos comunes:**
1. **Datos personales:** Email, nombre, información de contacto (ej: Google comparte números de teléfono con comercios tras interacciones con anuncios).
2. **Datos de dispositivo:** Identificadores (ID del dispositivo), sistema operativo, tipo de navegador.
3. **Datos de uso:** Actividad en el servicio (búsquedas, reproducciones), interacciones con publicidad, historial de navegación.
4. **Datos geográficos:** Ubicación precisa o aproximada, según permisos otorgados.
5. **Datos para análisis:** Uso de Google Analytics o herramientas similares para medir rendimiento de anuncios y optimizar servicios.
6. **Datos transferidos a terceros:** Proveedores de servicios colaboran en tareas como gestión de datos, seguridad (ej: análisis de audio en YouTube) o mantenimiento de servidores.

**Finalidades p

### 3.2. Preguntas específicas

Estas pruebas permiten analizar cómo el agente utiliza las tools de búsqueda de forma más flexible que en la Fase 2, especialmente en preguntas complejas o con múltiples partes.

In [62]:
responder_con_trazas("¿Qué requisitos hay que cumplir para usar Spotify?")
# El agente aplica los filtros y la query correcta y obtiene un resultado correcto, similar al de la fase 2.

Respuesta: Para utilizar Spotify, es necesario cumplir los siguientes requisitos según sus términos de servicio:

1. **Edad mínima**:  
   - Debe tener al menos 13 años (o la edad mínima equivalente en su país de origen).  
   - Si es menor de edad en su jurisdicción, se requiere el consentimiento de los padres o tutores legales para registrarse.

2. **Capacidad legal**:  
   - Debe ser titular de una personalidad jurídica (persona física o jurídica) que pueda celebrar contratos vinculantes de acuerdo con la legislación aplicable.

3. **Residencia**:  
   - Debe residir en un país donde Spotify presta sus servicios.

4. **Información exacta**:  
   - Se compromete a proporcionar información de registro precisa y actualizada.

5. **Aceptación de términos**:  
   - Al acceder al servicio, acepta tácitamente sus [Términos y Condiciones](https://www.spotify.com/).  
   - Los términos adicionales (como los detallados en el artículo 7) son parte integrante del acuerdo.

6. **Política de priv

In [ ]:
responder_con_trazas("¿Puede Google bloquear una cuenta de usuario?")
# Resultado correcto similar al de la fase 2.

Respuesta: Google puede bloquear o suspender una cuenta de usuario bajo ciertas circunstancias, según detallan los **términos de servicio de Google**:

1. **Incumplimiento grave o repetido**: Si el usuario viola los términos de servicio o políticas aplicables.
2. **Cumplimiento legal**: Cuando se requiere actuar conforme a una orden judicial o requerimiento legal.
3. **Daño perjudicial**: Si se cree de buena fe que el uso de los servicios causa daño.

Adicionalmente, Google podrá actualizar sus términos y, en caso de cambios sustanciales, notificará al usuario con anticipación razonable. Si el usuario no acepta las actualizaciones, debe dejar de usar los servicios. Los usuarios también tienen derecho a apelar si consideran que su cuenta fue bloqueada de forma incorrecta.

En organizaciones, un administrador autorizado puede gestionar cuentas de usuarios, incluyendo su deshabilitación.
> Thinking:    Okay, the user is asking if Google can block a user account. I need to check the terms 

In [ ]:
responder_con_trazas("¿Qué derechos tienen los usuarios sobre sus datos según el GDPR y cómo pueden ejercerlos en Spotify?")
# A diferencia de la fase 2, ahora si recupera chunks del documento 'gdpr_explained.md' y no solo 'spotify_privacy_policy.md'.
# Esto se debe a que primero hace una busqueda para responder a la primera parte de la pregunta y luego otra para responder a la segunda.

Respuesta: **Derechos de los usuarios según el GDPR y ejercicio en Spotify**  

### **Derechos reconocidos por el GDPR**  
1. **Derecho a ser informado**: Recibir transparencia sobre el uso de sus datos.  
2. **Acceso**: Solicitar copias de sus datos personales.  
3. **Rectificación**: Corregir datos inexactos.  
4. **Borrado**: Solicitar la eliminación de datos cuando sea pertinente.  
5. **Límite de procesamiento**: Restringir el uso de datos en ciertos casos.  
6. **Transportabilidad**: Pedir su información en formato electrónico y transferirla a otros servicios.  
7. **Oposición**: Rechazar el procesamiento de datos, especialmente para fines de publicidad personalizada.  
8. **No ser objeto de decisiones automatizadas**: Exigir que no se tomen decisiones sin intervención humana.  

### **Cómodamente en Spotify**  
1. **Oposición**:  
   - Utilizar los controles del servicio (ej.: ajustes de publicidad) para desactivar funciones que usan sus datos.  
   - Contactar directamente a Sp

In [ ]:
responder_con_trazas("¿Qué edad mínima se requiere para usar Spotify? ¿Y para usar Google?")
# Aqui el agente toma una estrategia mejor que en la fase 2: hace una búsqueda para cada parte de la pregunta.
# Después, detecta que en los términos de Google no se especifica una edad mínima y el modelo responde con informacion general de su propio conocimiento.

Respuesta: Según los términos oficiales consultados:

1. **Spotify**: La edad mínima requerida es de **13 años** (o la edad equivalente según la legislación del país del usuario). Menores de esa edad necesitarán el consentimiento de sus padres o tutores legales.

2. **Google**: Aunque no se establece explícitamente una edad en los términos generales, se menciona que **menores de la edad legal en su país** deben obtener autorización parental para usar servicios de Google. En la práctica, Google aplica la normativa COPPA de EE.UU., que fija **13 años** como edad mínima para crear una cuenta.

En ambos casos, los requisitos pueden variar según la jurisdicción.
> Thinking:    Okay, I need to find the minimum age required to use Spotify and Google. Let me start by recalling if I know this information off the top of my head. I remember that many online services have a minimum age, often 13, due to regulations like COPPA in the US. But to be sure, I should check the official terms of service 

In [ ]:
responder_con_trazas("Which is more privacy-friendly: Google or Spotify?")
# Hace una busqueda para recuperar la informacion de Google y otra para Spotify.
# De esta forma, da una respuesta completa comparando ambos resultados al tener un mayor contexto, 
# a diferencia de la fase 2 donde no conseguia realizar la comparación porquesolo recuperaba chunks de Google.

Respuesta: Based on the privacy policies of **Google** and **Spotify**, here is a structured comparison:

---

### **Key Privacy Differences**

| **Aspect**                | **Google**                                                                 | **Spotify**                                                                 |
|--------------------------|-----------------------------------------------------------------------------|------------------------------------------------------------------------------|
| **Data Collection Scope** | Broad data collection across services (e.g., search history, location, device info) and third-party integrations. | Focused data collection related to music preferences, playlists, and device usage. |
| **Third-Party Sharing**   | Shares personal data with partners for advertising, analytics, and legal obligations. Requires explicit consent for sensitive data. | Shares data with third-party apps/users only with explicit consent (e.g., Spotify Connect)

### 3.3. Preguntas sobre información muy concreta

Con estas preguntas se busca evaluar la capacidad del RAG para recuperar información muy específica que aparece en una sola línea o pocas líneas de un documento concreto.

In [ ]:
responder_con_trazas("¿Cómo puedo ponerme en contacto con Spotify si tengo dudas sobre su política de privacidad?")
# Recupera el chunk específico de spotify_privacy.md y responde correctamente a la pregunta

Respuesta: Si tienes dudas sobre la política de privacidad de Spotify, puedes ponerte en contacto con ellos de las siguientes maneras:  

1. **Correo electrónico**: Envía un mensaje a **privacy@spotify.com**.  
2. **Dirección postal**:  
   Spotify AB  
   Regeringsgatan 19  
   111 53 Estocolmo, Suecia.  

Además, si necesitas acceder a tus datos personales, puedes:  
- Usar la herramienta **"Descarga de tus datos"** en la página **Privacidad de la cuenta** de tu perfil de Spotify.  

Estos métodos son los indicados en su política de privacidad para gestionar consultas o solicitudes relacionadas con el tratamiento de datos.
> Thinking:    Okay, the user is asking how to contact Spotify if they have questions about their privacy policy. I remember that privacy policies usually have a contact section for such inquiries. Let me check if I can find the specific contact information from Spotify's privacy policy using the search_documents tool. The user might need details like email, phone 

In [ ]:
responder_con_trazas("¿Cómo puedo ponerme en contacto con Google desde la Polinesia Francesa para ejercer mi derecho de desistimiento?")
# Recupera correctamente el chunk relevante y responde correctamente a la pregunta

Respuesta: Para ejercer su derecho de desistimiento en la Polinesia Francesa y ponerse en contacto con Google, puede utilizar los siguientes canales oficiales:

1. **Teléfono**: Llame al **+33 1 85 14 96 65** (número disponible para Francia y sus territorios ultramarinos como la Polinesia Francesa).  
2. **Correo electrónico**: Envíe un mensaje a **account-withdrawal@google.com**.  
3. **Correo postal**: Dirección: Google Ireland Limited, Gordon House, Barrow Street, Dublin 4, Irlanda.  

Además, debe informar su desistimiento de forma inequívoca (por ejemplo, mediante una carta o correo electrónico). Según los términos, el plazo de desistimiento es de **14 días desde la celebración del contrato**. Tenga en cuenta que Google no está obligada a utilizar la Plataforma de Resolución de Disputas Online de la UE.
> Thinking:    Okay, the user is asking how to contact Google from French Polynesia to exercise their right of withdrawal. First, I need to check the specific terms or privacy poli

In [ ]:
responder_con_trazas("¿Desde cuando está en vigor la política de privacidad de Spotify? ¿Y sus términos de servicio?")
# A diferencia de la fase 2, ahora sí consigue recuperar chunks relevantes para responder a ambas preguntas 
# al realizar búsquedas separadas para cada una.

Respuesta: La **política de privacidad** de Spotify está en vigor desde el **27 de agosto de 2025**, según consta en los documentos oficiales.  

Los **términos de servicio** fueron actualizados por última vez el **26 de agosto de 2025**, y este documento generalmente se considera vigente desde la fecha de su última actualización, a menos que se indique otra fecha específica.  

En resumen:  
- **Política de privacidad**: Vigente desde el **27/08/2025**.  
- **Términos de servicio**: Vigentes desde el **26/08/2025** (última actualización).
> Thinking:    Okay, the user is asking when Spotify's privacy policy and terms of service went into effect. Let me break this down.

First, I need to find the effective dates for both documents. The question specifies "política de privacidad" (privacy policy) and "términos de servicio" (terms of service), so I should check both. 

I remember that the search_documents tool can look up official documents. For the privacy policy, I should use categoría

### 3.4. Preguntas fuera de dominio

In [ ]:
responder_con_trazas("¿Cómo funciona la fotosíntesis de las plantas?")
# No usa ninguna tool y responde brevemente e indica que no es un tema de su dominio

Respuesta: La fotosíntesis es un proceso biológico relacionado con la producción de energía en plantas, no con privacidad o protección de datos. Este tema está fuera de mi dominio de expertise, que se centra en normativas de privacidad, políticas de datos y términos de servicio de empresas. ¿Tienes preguntas sobre estos temas?
> Thinking:    Okay, the user is asking about how photosynthesis works in plants. Let me check if this is within my domain of privacy and data protection. The question is about a biological process, which doesn't relate to data privacy, GDPR, or company policies. The tools I have are for searching documents related to privacy policies, terms of service, or guides on data protection. Since the user's query is about plant biology, none of the provided functions would be relevant here. I can't use search_documents because the topic isn't covered in their corpus. Web_search is also restricted to when search_documents isn't sufficient, but even then, the user's questi

In [ ]:
responder_con_trazas("¿Cuál es la capital de Francia?")
# Igual que antes, responde que no es un tema de su dominio y no intenta usar ninguna herramienta para responder.

Respuesta: La pregunta sobre la capital de Francia no se relaciona con el ámbito de privacidad y protección de datos, que es el estrictamente autorizado. En ese caso, no puedo ofrecer información sobre este tema. Si tienes dudas sobre privacidad, datos personales o términos de servicio de plataformas como Google o Spotify, estaré encantado de ayudarte.
> Thinking:    Okay, the user is asking for the capital of France. Let me check if this is related to privacy or data protection. The question is about geography, so it's outside my domain. The tools provided are for searching documents related to privacy, terms of service, and web searches, but only if the first tool doesn't yield enough results. Since the user's query isn't in the domain, I should decline to answer. I need to make sure I don't provide any information outside the specified areas. So, I'll inform them that I can't help with that question.

> Respuesta:   La pregunta sobre la capital de Francia no se relaciona con el ámbi

### 3.5. Preguntas relacionadas con el dominio pero que no estan en el corpus y preguntas trampa

Estas pruebas se diseñaron para analizar cómo el agente gestiona situaciones donde la información solicitada no aparece en el corpus, así como preguntas formuladas para intentar provocar alucinaciones o razonamientos incorrectos.

In [ ]:
responder_con_trazas("¿Qué datos recopila Netflix cuando usamos sus servicios?")
# En este caso, tarda bastante más en responder.
# Primero intenta usar 'search_documents', pero como no obtienen información relevante (porque no hay documentos de Netflix en el corpus), 
# el agente decide usar 'web_search' para recuperar información y respone correctamente.

Respuesta: Netflix recopila diversos tipos de datos al usar sus servicios, según sus políticas de privacidad y fuentes externas verificadas:

1. **Datos personales**:  
   - **Información básica**: Nombre, dirección de correo electrónico, número de teléfono, dirección postal y datos de pago (tarjetas, métodos de factura).  
   - **Geolocalización**: Ubicación general y región específica (ej. "en el Reino Unido").  

2. **Datos técnicos y de dispositivos**:  
   - Identificadores de dispositivos (ej. ID de Apple TV, Samsung TV, iPad).  
   - Tipo de navegador, sistema operativo y características técnicas del dispositivo.  
   - Historial de dispositivos usados para acceder a Netflix (ej. MacBook Pro, smartphone).  

3. **Datos de uso y comportamiento**:  
   - Series y películas vistas, interacciones con el servicio (pausas, búsquedas).  
   - Datos de cookies y tecnologías similares para personalizar contenido y publicidad.  
   - Tiempo de visualización y patrones de consumo.  

4. **

In [ ]:
responder_con_trazas("¿Cómo puedo crear una cuenta en Spotify?")
# Igual que con la pregunta anterior, primero hace una búsqueda con 'search_documents', pero como no obtiene información relevante,
# el agente decide usar 'web_search' para recuperar información y responde correctamente.

Respuesta: Para crear una cuenta en Spotify, sigue estos pasos principales:

1. **Descarga la aplicación**: 
   - Instala la aplicación Spotify en tu dispositivo (iOS o Android) desde las respectivas tiendas de aplicaciones.

2. **Regístrate**:
   - Abre la app y selecciona la opción "Registrarse" (o "Sign up").
   - Introduce tu correo electrónico, nombre de usuario y una contraseña.

3. **Verificación**:
   - Confirma tu dirección de correo electrónico a través del código de verificación enviado a tu correo.

4. **Iniciar sesión**:
   - Una vez verificado, inicia sesión con tus credenciales para acceder a Spotify.

**Alternativa vía navegador**:  
También puedes crear una cuenta desde el sitio web oficial de Spotify ([https://www.spotify.com](https://www.spotify.com)) siguiendo un proceso similar.

Los tutoriales en YouTube y Reddit confirman que el proceso es estándar para dispositivos móviles y navegadores, y recomiendan verificar el correo electrónico como paso obligatorio. Para o

In [ ]:
responder_con_trazas("¿Qué requisitos tiene que cumplir un gato para usar Spotify?")
# Al igual que en la fase 2, el agente identifica correctamente el elemento absurdo (gato) indicando que la politica es solo para usuarios humanos.

Respuesta: Un gato no puede cumplir los requisitos para usar Spotify, ya que los términos de servicio de la plataforma establecen condiciones que solo un ser humano puede aceptar y gestionar. Estos incluyen:

1. **Registro y consentimiento**: Crear una cuenta requiere proporcionar un correo electrónico válido y aceptar los términos de servicio y la política de privacidad, lo cual implica comprensión y asentimiento consciente.

2. **Edad mínima**: El usuario debe tener al menos 18 años o la edad de mayoría en su jurisdicción (si es mayor de 18 años).

3. **Método de pago**: Para suscripciones Premium, se necesita un método de pago válido (tarjeta de crédito, débito, etc.), que un gato no puede administrar.

4. **Uso personal y legítimo**: El servicio debe usarse de conformidad con las directrices de Spotify, lo que exige un control humano sobre el acceso y la actividad en la cuenta.

Espotify no tiene disposiciones específicas para animales: los términos establecen explícitamente que el

In [ ]:
responder_con_trazas("¿Qué métodos de encriptado utiliza Google para proteger los datos de los usuarios?")
# Al igual que en la fase 2, responde que no tiene informacion específica sobre métodos de encriptado,
# lo cual es correcto porque no aluciona una respuesta. Sin embargo, no decide usar la herramienta 'web_search' después.

Respuesta: Google utiliza encriptación para proteger los datos de los usuarios mientras se transmiten (en tránsito), como se indica en su Política de Privacidad. Además, menciona en forma general que:

1. **Encriptación en tránsito**: Datos sensibles están encriptados durante la transmisión a través de la red.
2. **Seguridad integrada**: Sus servicios incluyen funciones de seguridad automatizadas para detectar y bloquear amenazas, aunque no se detallan algoritmos específicos de encriptación en los documentos revisados.

La información disponible no especifica métodos adicionales (como AES o TLS) ni encriptación de datos en reposo. Para detalles técnicos más precisos, se recomendaría consultar documentación técnica específica de Google o su infraestructura de seguridad.
> Thinking:    Okay, the user is asking what encryption methods Google uses to protect user data. I need to find this information. First, I should check if there are official documents from Google about their encryption 

In [ ]:
responder_con_trazas("How does Google sell user passwords to advertisers?")
# Identifica correctamente el supuesto falso.

Respuesta: According to Google's Privacy Policy and Terms of Service:

1. **Data Usage for Advertising**: Google analyzes user activity (e.g., YouTube video views) to serve targeted ads. This involves associating user behavior with accounts to improve ad relevance, but explicitly excludes sharing sensitive data like passwords.

2. **Security Measures**: Google states they protect user information from unauthorized access and do not "sell" personal data per GDPR definitions. Data sharing with third parties is limited to advertising partnerships and occurs under contractual obligations to comply with privacy laws.

3. **Abuse Detection**: Automated systems analyze content to detect fraud or abuse (e.g., spam, malware), but this does not involve transferring sensitive credentials to advertisers.

In conclusion, Google's policies confirm they do not sell user passwords to advertisers. Advertising personalization occurs through anonymized behavioral analysis, not direct data transfer of sen

In [ ]:
# Pregunta vaga para ver si el modelo alucina información concreta.
responder_con_trazas("What happens with my data?")
# Igual que en la fase 2, responde solo con los documentos disponibles, sin alucinar

Respuesta: Based on official documents, here’s how your data is typically handled under privacy regulations like GDPR and by companies such as Google:

1. **Security Measures**:
   - **Technical protections**: Encryption, two-factor authentication, secure cloud storage.
   - **Organizational safeguards**: Staff training, limited data access, privacy policies.
   - **Breach notifications**: Companies must report data breaches within 72 hours (unless encryption renders data unusable).

2. **Legal Principles**:
   - **Transparency**: You must be informed about data collection purposes.
   - **Data Minimization**: Only essential data is collected.
   - **Purpose Limitation**: Data is used exclusively for the stated purpose.
   - **Storage Limitation**: Data is deleted once no longer needed.
   - **Accuracy & Integrity**: Data must be accurate and protected from misuse.

3. **User Control**:
   - Ability to update, access, delete, or export your data (as outlined in Google's Privacy Policy)

In [ ]:
responder_con_trazas("If cookies are not used for authentication, why do websites use them?")
# Igual que en la fase 2, en el razonamiento (thinking) detecta la contradicción e indica en la respuesta otros fines para lo que se utilizan las cookies.

Respuesta: Websites use cookies for various purposes beyond authentication, including:

1. **User Preferences**: Storing user settings (e.g., language, theme, or font size) to personalize the browsing experience.  
2. **Session Management**: Maintaining user state within a session (e.g., items in a shopping cart) without constant re-authentication.  
3. **Analytics**: Tracking user behavior (e.g., page visits, clicks) to improve site performance and user experience.  
4. **Advertising**: Delivering targeted ads based on browsing history or interests.  
5. **Functionality**: Enabling features like form pre-filling or location-based services.  

Cookies ensure websites operate efficiently and provide a tailored experience without relying on authentication mechanisms.
> Thinking:    Okay, the user is asking why websites use cookies if they aren't used for authentication. Let me break this down.

First, I know cookies have multiple purposes beyond authentication. Authentication is one use,

In [ ]:
responder_con_trazas("Complete the sentence: 'Spotify completely disregards users' rights because...'")
# Responde igual que en la fase 2: detecta el supuesto falso y lo contradice.

Respuesta: La afirmación de que "Spotify completamente desprecia los derechos de los usuarios" es incorrecta según su política de privacidad vigente. Spotify concede derechos a los usuarios como:

1. **Acceso a datos personales** vía la herramienta "Descarga de tus datos" en la página de privacidad de la cuenta.
2. **Portabilidad de datos** permitiendo la copia de información en formato electrónico y su transferencia a otros servicios.
3. **Prohibición de decisiones automatizadas** con impacto legal, no aplicándose en su servicio.

La política especifica canales claros (sitio web, soporte) para ejercer estos derechos, cumpliendo con principios del RGPD. No hay indicios en los documentos proporcionados de negligencia en el tratamiento de derechos usuarios.
> Thinking:    Okay, the user wants to complete the sentence "Spotify completely disregards users' rights because..." I need to figure out the reason based on their privacy policy or terms of service. First, I should check Spotify's p

### 3.6. Preguntas para probar las tools auxiliares

In [ ]:
responder_con_trazas("¿Cuántos años han pasado desde que entró en vigor la política de privacidad de Spotify?")
# Primero recupera la fecha de entrada en vigor a través de la herramienta 'search_documents' y luego calcula 
# correctamente los años transcurridos usando la herramienta 'get_current_date' para obtener la fecha actual. 

Respuesta: La política de privacidad de Spotify entró en vigor el 27 de agosto de 2025. Dado que la fecha actual es 7 de abril de 2026, han transcurrido **7 meses y 11 días** desde su entrada en vigor. Por lo tanto, **aún no han pasado un año completo** (0 años y 7 meses aproximadamente). 

Si se calcula en términos estrictamente anuales, el tiempo transcurrido es **menos de un año**.
> Thinking:    Okay, the user is asking how many years have passed since Spotify's privacy policy went into effect. I need to figure out the current date and the effective date of the policy.

First, I'll use the get_current_date function to get today's date. The current date is 30 de noviembre de 2023.

Next, I need to find the effective date of Spotify's current privacy policy. I should check Spotify's official privacy policy document. The right tool here is search_documents with the parameters empresa set to Spotify and categoria as privacy_policies. I'll search for the effective date in the privacy po

In [ ]:
responder_con_trazas("Teniendo en cuenta que una empresa factura 2 millones de euros anuales, ¿cuánto tendría que pagar de multa si incumple el GDPR?")
# Primero busca en el corpus cual es el porcentaje máximo de multa que se puede aplicar por incumplimiento del GDPR 
# y luego hace el cálculo correcto usando la herramienta 'calculator'.

Respuesta: La multa máxima por incumplir el GDPR para una empresa con facturación anual de 2 millones de euros es del **4% de su facturación**, es decir, **80.000 euros**.

### Detalles:
- **Mecanismo de cálculo:** El GDPR establece multas de hasta **4% de la facturación anual global** (o **20 millones de euros**, lo que sea mayor). 
- **Caso concreto:** Dado que 4% de 2 millones de euros es **80.000 euros**, este sería el importe máximo aplicable.
- **Contexto:** Este monto corresponde a infracciones graves (por ejemplo, violaciones de obligaciones esenciales sobre consentimiento o privacidad por diseño). Infracciones menos graves podrían generar multas de hasta **2% de la facturación** (es decir, 40.000 euros). 

Esta información se desprendió de fuentes oficiales sobre la estructura sancionatoria del GDPR.
> Thinking:    Okay, let's tackle this question. The user wants to know the potential GDPR fine for a company that invoices 2 million euros annually. First, I need to recall the G